In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/hate-speech-detection-vietnamese/val_df.csv
/kaggle/input/hate-speech-detection-vietnamese/train_df.csv
/kaggle/input/hate-speech-detection-vietnamese/test_df.csv


In [2]:
!pip install pyvi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 61.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.5 MB/s eta 0:00:00:00:01


In [3]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pyvi import ViTokenizer, ViPosTagger, ViUtils
from datasets import Dataset

In [27]:
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base", num_labels=3)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.pooler.dense.weight     | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initia

In [5]:
test_df  = pd.read_csv("/kaggle/input/hate-speech-detection-vietnamese/test_df.csv")
train_df = pd.read_csv("/kaggle/input/hate-speech-detection-vietnamese/train_df.csv")
val_df   = pd.read_csv("/kaggle/input/hate-speech-detection-vietnamese/val_df.csv")

train_df['labels'] = train_df['labels'].astype(np.int64)
test_df['labels']  = test_df['labels'].astype(np.int64)
val_df['labels']   = val_df['labels'].astype(np.int64)

print(f"Độ dài của tập test: {len(test_df)}")
print(f"Độ dài của tập train: {len(train_df)}")
print(f"Độ dài của tập val: {len(val_df)}")

Độ dài của tập test: 1476
Độ dài của tập train: 5166
Độ dài của tập val: 738


In [6]:
# ViTokenizer, ViPosTagger, ViUtils
def tokenize_text(row):
    # tokenizer nhận , nhưng ViTokenizer thì không nên phải xử lý từng chữ
    text = [ViTokenizer.tokenize(text) for text in row['cmt_col']]
    return tokenizer(text, padding="max_length", truncation=True, max_length=100)

In [7]:
train_data = Dataset.from_pandas(train_df)
test_data = Dataset.from_pandas(test_df)
val_data = Dataset.from_pandas(val_df)

train_data = train_data.map(tokenize_text, batched=True)
test_data = test_data.map(tokenize_text, batched=True)
val_data = val_data.map(tokenize_text, batched=True)

train_data

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Map:   0%|          | 0/5166 [00:00<?, ? examples/s]

Map:   0%|          | 0/1476 [00:00<?, ? examples/s]

Map:   0%|          | 0/738 [00:00<?, ? examples/s]

Dataset({
    features: ['cmt_col', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 5166
})

In [26]:
!rm -r checkpoints
!rm -r /kaggle/working/hate-speech-model

rm: cannot remove '/kaggle/working/hate-speech-model': No such file or directory


In [28]:
train_args = TrainingArguments(
    output_dir = "checkpoints",
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    learning_rate = 3e-5,
    num_train_epochs = 8,
    eval_strategy = "epoch",
    logging_strategy = "epoch",
    weight_decay = 0.001
)

trainer = Trainer(
    args = train_args,
    model = model,
    train_dataset = train_data,
    eval_dataset = test_data
)

trainer.train()
model.save_pretrained("./hate-speech-model")
tokenizer.save_pretrained("./hate-speech-model")

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,1.768333,1.304486
2,1.213331,1.126779
3,0.881136,1.136409
4,0.663862,1.213047
5,0.478770,1.238444
6,0.375794,1.288677
7,0.272972,1.370988
8,0.214798,1.462444


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./hate-speech-model/tokenizer_config.json',
 './hate-speech-model/vocab.txt',
 './hate-speech-model/bpe.codes',
 './hate-speech-model/added_tokens.json')

In [29]:
from sklearn.metrics import classification_report
predictions = trainer.predict(val_data)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = val_data['labels']
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.84      0.86       290
           1       0.73      0.74      0.73       253
           2       0.79      0.84      0.82       195

    accuracy                           0.80       738
   macro avg       0.80      0.80      0.80       738
weighted avg       0.81      0.80      0.80       738



In [10]:
new_model = AutoModelForSequenceClassification.from_pretrained("/kaggle/working/hate-speech-model")
new_tokenizer = AutoTokenizer.from_pretrained("/kaggle/working/hate-speech-model")
device = torch.device("cuda")
new_model.to(device)
new_model.eval()
text = "địt mẹ bạn xàm lồn vừa thôi"
with torch.no_grad():
    vi_text = ViTokenizer.tokenize(text)
    tokens = new_tokenizer(vi_text, padding="max_length", max_length=100, truncation=True, return_tensors='pt').to(device)
    outputs = model(**tokens)
    print(vi_text)
    print(outputs)
    print(f"Class: {np.argmax(outputs.logits.cpu())}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

địt_mẹ bạn xàm lồn vừa thôi
SequenceClassifierOutput(loss=None, logits=tensor([[-2.5895,  0.3040,  2.1668]], device='cuda:0'), hidden_states=None, attentions=None)
Class: 2


In [30]:
from huggingface_hub import upload_folder, HfApi
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("HF_TOKEN")

api = HfApi(token=api_key)
api.upload_folder(
    folder_path="hate-speech-model",
    repo_id="VietAnh-1027/hate-speech-model",
    repo_type="model"
)
print("Đẩy model lên hugging face hub thành công!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Đẩy model lên hugging face hub thành công!
